In [1]:
import pandas as pd

# 1. Read raw data
df_raw = pd.read_csv('ACLED_data.csv', parse_dates=['event_date'])

print(f"Number of original data rows: {len(df_raw)}")
print(f"Original countries included: {df_raw['country'].unique()} \n")

# =========================================================
# Dealing with the country names
# =========================================================
country_mapping = {
    'England': 'United Kingdom',
    'Scotland': 'United Kingdom',
    'Wales': 'United Kingdom',
    'Northern Ireland': 'United Kingdom',
    'Czechia': 'Czech Republic',
    'Czech Republic': 'Czech Republic'
}

# Define the final list of countries that want to retain
target_countries = ['Germany', 'France', 'United Kingdom', 'Czech Republic', 'Slovakia']

df_raw['country'] = df_raw['country'].replace(country_mapping)

# =========================================================
# Data filtering logic
# =========================================================
df_filtered = df_raw[df_raw['country'].isin(target_countries)].copy()

# Keep only the "protest and demonstration" events and eliminate all armed conflicts and riots
df_filtered = df_filtered[df_filtered['event_type'] == 'Protests']

# Only retain data with a time precision of 1 (accurate to a certain day) to prevent dirty data from occurring during monthly aggregation
df_filtered = df_filtered[df_filtered['time_precision'] == 1]

# =========================================================
# Simplify Columns
# =========================================================
# Retain the columns that truly requires (excluding news sources, redundant timestamps, etc.)
columns_to_keep = [
    'event_id_cnty', 'event_date', 'year', 
    'event_type', 'sub_event_type', 
    'actor1', 'actor2', 'interaction',
    'country', 'admin1', 'admin2', 'location', 'latitude', 'longitude',
    'fatalities', 
    'tags',
    'notes', 
    'population_1km'
]

df_cleaned = df_filtered[columns_to_keep]

# =========================================================
# Final inspection and export
# =========================================================
df_cleaned = df_cleaned.reset_index(drop=True)

print(f"Number of rows of data after cleaning: {len(df_cleaned)}")
print(f"Currently included countries: {df_cleaned['country'].unique()}")
print(f"Current event type: {df_cleaned['event_type'].unique()}")
print(f"Distribution of sub event types (focus on Excess force):\n{df_cleaned['sub_event_type'].value_counts()} \n")

df_cleaned.to_csv('ACELD_analysis_dataset.csv', index=False)

Number of original data rows: 211043
Original countries included: ['Albania' 'Belarus' 'Bosnia and Herzegovina' 'Bulgaria' 'Greece' 'Russia'
 'Serbia' 'Ukraine' 'Cyprus' 'Moldova' 'Romania' 'Kosovo' 'Croatia'
 'Andorra' 'Czech Republic' 'Germany' 'Denmark' 'Estonia' 'Finland'
 'France' 'United Kingdom' 'Greenland' 'Hungary' 'Isle of Man' 'Ireland'
 'Italy' 'Latvia' 'Norway' 'Poland' 'Portugal' 'Slovakia' 'Slovenia'
 'Sweden' 'Spain' 'Lithuania' 'Netherlands' 'Vatican City' 'Belgium'
 'Luxembourg' 'Faroe Islands' 'Monaco' 'Iceland' 'San Marino'
 'Switzerland' 'North Macedonia' 'Montenegro' 'Austria'
 'Bailiwick of Guernsey' 'Bailiwick of Jersey' 'Liechtenstein' 'Malta'
 'Akrotiri and Dhekelia' 'Gibraltar'] 

Number of rows of data after cleaning: 68676
Currently included countries: ['Czech Republic' 'Germany' 'France' 'United Kingdom' 'Slovakia']
Current event type: ['Protests']
Distribution of sub event types (focus on Excess force):
sub_event_type
Peaceful protest                     

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('ACELD_analysis_dataset.csv', parse_dates=['event_date'])

# ==========================================
# 1. Redefine "state violence / coercive intervention" (numerator)
# ==========================================
# Treat "Excessive force" and "Protest with intervention" as forms of state repression
repressive_types = ['Protest with intervention', 'Excessive force against protesters']

# Create a new binary indicator: 1 if the event involved repression, 0 otherwise
df['is_repressed'] = df['sub_event_type'].isin(repressive_types).astype(int)

# ==========================================
# 2. Extract year-month
# ==========================================
df['year_month'] = df['event_date'].dt.to_period('M')

# ==========================================
# 3. Core step: build the panel dataset
# ==========================================
print("Generating country-month panel data...")

# Group by country and month
panel_df = df.groupby(['country', 'year_month']).agg(
    total_protests=('event_id_cnty', 'count'),     
    repressed_events=('is_repressed', 'sum'),    
    avg_fatalities=('fatalities', 'mean')   
).reset_index()

# ==========================================
# 4. Compute the core dependent variable: crackdown rate
# ==========================================
# Avoid division-by-zero errors: if a country had no protests in a given month,
# set the crackdown rate to NaN (it will be dropped automatically in later regression models)
panel_df['crackdown_rate'] = np.where(
    panel_df['total_protests'] > 0,
    panel_df['repressed_events'] / panel_df['total_protests'],
    np.nan
)

# Convert period type back to string for easier export and later merging with V-Dem
panel_df['year_month'] = panel_df['year_month'].astype(str)

# ==========================================
# 5. Inspect the output
# ==========================================
print("Panel dataset successfully generated!")
print(f"Total observations (country-month): {len(panel_df)}")
print("---------------- Preview: first 10 rows ----------------")
print(panel_df.head(10))

print("\n---------------- Key descriptive statistics ----------------")
print(f"Average monthly protest count: {panel_df['total_protests'].mean():.2f}")
print(f"Average monthly repressed events: {panel_df['repressed_events'].mean():.2f}")
print(f"Average crackdown rate: {panel_df['crackdown_rate'].mean():.4f} ({panel_df['crackdown_rate'].mean()*100:.2f}%)")

# Export the panel dataset
panel_df.to_csv('ACLED_Panel_Monthly.csv', index=False)

Generating country-month panel data...
Panel dataset successfully generated!
Total observations (country-month): 319
---------------- Preview: first 10 rows ----------------
          country year_month  total_protests  repressed_events  \
0  Czech Republic    2020-01               9                 0   
1  Czech Republic    2020-02              13                 1   
2  Czech Republic    2020-03               5                 0   
3  Czech Republic    2020-04               6                 0   
4  Czech Republic    2020-05              15                 1   
5  Czech Republic    2020-06              66                 0   
6  Czech Republic    2020-07              11                 1   
7  Czech Republic    2020-08              17                 0   
8  Czech Republic    2020-09              16                 1   
9  Czech Republic    2020-10               9                 1   

   avg_fatalities  crackdown_rate  
0             0.0        0.000000  
1             0.0        0.

In [5]:
import pandas as pd

df = pd.read_csv('ACELD_analysis_dataset.csv', parse_dates=['event_date'])

# ==========================================
# 1. Add a repression flag
# This allows the map to distinguish events by color
# based on whether they involved repression/intervention
# ==========================================
repressive_types = ['Protest with intervention', 'Excessive force against protesters']
df['is_repressed'] = df['sub_event_type'].isin(repressive_types).astype(int)

# ==========================================
# 2. Aggregate at the city level
# ==========================================
# Use country + location + latitude + longitude as a unique identifier.

city_map_df = df.groupby(['country', 'location', 'latitude', 'longitude']).agg(
    total_protests=('event_id_cnty', 'count'),   
    repressed_count=('is_repressed', 'sum')     
).reset_index()

# Sort by total protest count in descending order to inspect the most active protest locations
city_map_df = city_map_df.sort_values(by='total_protests', ascending=False)

print(f"A total of {len(city_map_df)} unique protest locations (cities/neighborhoods) were aggregated.")
print(f"Top 5 locations by protest count:\\n{city_map_df.head()}")

# ==========================================
# 3. Export map-ready data
# ==========================================
city_map_df.to_csv('ACLED_City_Map_Data.csv', index=False)

A total of 7357 unique protest locations (cities/neighborhoods) were aggregated.
Top 5 locations by protest count:\n             country        location  latitude  longitude  total_protests  \
2657          France           Paris   48.8566     2.3522            1725   
4216         Germany  Berlin - Mitte   52.5177    13.4024             936   
2247          France       Marseille   43.2965     5.3698             703   
3686          France        Toulouse   43.6047     1.4442             684   
124   Czech Republic          Prague   50.0875    14.4213             643   

      repressed_count  
2657              134  
4216               84  
2247               16  
3686               32  
124                38  


In [7]:
import pandas as pd

# ==========================================
# 1. Load micro-level event data (with lat/lon, city names)
# ==========================================
print("Loading micro-level event data...")
df_micro = pd.read_csv('ACELD_analysis_dataset.csv', parse_dates=['event_date'])

# Extract year to serve as the bridge for merging with V-Dem
df_micro['year'] = df_micro['event_date'].dt.year

# ==========================================
# 2. Safely load V-Dem data (using your verified variables)
# ==========================================
print("Safely loading V-Dem v16...")
vdem_cols_to_keep = [
    'country_name', 'year', 
    'v2x_polyarchy', 'v2x_libdem', 'v2x_freexp_altinf', 
    'v2x_rule', 'v2x_regime', 'v2x_partipdem', 
    'v2x_delibdem', 'v2x_egaldem'
]
df_vdem = pd.read_csv('V-Dem-CY-Full+Others-v16.csv', usecols=vdem_cols_to_keep, low_memory=False)

# ==========================================
# 3. Country name alignment
# ==========================================
vdem_country_mapping = {
    'Germany': 'Germany',
    'France': 'France',
    'Great Britain': 'United Kingdom',  
    'United Kingdom': 'United Kingdom',
    'Czechia': 'Czech Republic',        
    'Czech Republic': 'Czech Republic',
    'Slovakia': 'Slovakia'
}
df_vdem['country'] = df_vdem['country_name'].replace(vdem_country_mapping)

# ==========================================
# 4. Perform left join at the micro-level
# ==========================================
print("Anchoring annual V-Dem data to each micro-level protest event...")

# Select only the V-Dem columns needed for merging
vdem_merge_cols = ['country', 'year', 'v2x_polyarchy', 'v2x_libdem', 
                   'v2x_freexp_altinf', 'v2x_rule', 'v2x_regime', 
                   'v2x_partipdem', 'v2x_delibdem', 'v2x_egaldem']

merged_micro_df = pd.merge(
    df_micro,
    df_vdem[vdem_merge_cols],
    on=['country', 'year'],
    how='left'
)

# ==========================================
# 5. Handle 2025 missing values (critical step!)
# ==========================================
# Check missing values after merging
initial_nulls = merged_micro_df['v2x_libdem'].isnull().sum()
print(f"Number of missing values after merging: {initial_nulls}")

# Drop 2025 events that lack V-Dem data
merged_micro_df = merged_micro_df.dropna(subset=['v2x_libdem'])

# ==========================================
# 6. Final verification and export
# ==========================================
print("---------------- Micro-level fusion complete ----------------")
print(f"Final micro-level dataset size after cleaning 2025 events: {merged_micro_df.shape}")
print(f"All columns included: {merged_micro_df.columns.tolist()}")

# Export micro-level data with macro political context for mapping
merged_micro_df.to_csv('Micro_With_VDem.csv', index=False)

Loading micro-level event data...
Safely loading V-Dem v16...
Anchoring annual V-Dem data to each micro-level protest event...
Number of missing values after merging: 0
---------------- Micro-level fusion complete ----------------
Final micro-level dataset size after cleaning 2025 events: (68676, 26)
All columns included: ['event_id_cnty', 'event_date', 'year', 'event_type', 'sub_event_type', 'actor1', 'actor2', 'interaction', 'country', 'admin1', 'admin2', 'location', 'latitude', 'longitude', 'fatalities', 'tags', 'notes', 'population_1km', 'v2x_polyarchy', 'v2x_libdem', 'v2x_freexp_altinf', 'v2x_rule', 'v2x_regime', 'v2x_partipdem', 'v2x_delibdem', 'v2x_egaldem']
